# AI Functions: Ray + External Model on Les Mis

Run batch inference on the **Les Mis** dataset using **Ray** on Databricks to call an **external model** (OpenAI-compatible endpoint). Use this to test and benchmark external models alongside AI_QUERY and other ai_functions notebooks.

**Prerequisites:**
- Les Mis data prepared (run `les_mis_data_prep.ipynb` and optionally create endpoints via `les_mis_endpoint_creation.ipynb`).
- External model: OpenAI-compatible API (e.g. Azure OpenAI, or Databricks external model endpoint). Set `EXTERNAL_MODEL_BASE_URL` and `EXTERNAL_MODEL_API_KEY` (or use Databricks secrets).

In [ ]:
# Optional: install if not in cluster env
# %pip install ray openai --quiet
# %restart_python

## 1. Start Ray cluster on Spark (Databricks)

Uncomment `ray.util.spark.shutdown_ray_cluster()` below if you need to tear down an existing Ray cluster first.

In [ ]:
# ray.util.spark.shutdown_ray_cluster()

from ray.util.spark import setup_ray_cluster
import ray

setup_ray_cluster(
    min_worker_nodes=0,
    max_worker_nodes=3,
    num_cpus_head_node=0,
    num_gpus_worker_node=0,
    num_cpus_worker_node=4,
    num_gpus_head_node=0,
)

ray.init(ignore_reinit_error=True)

## 2. Load Les Mis dataset

Use the table from data prep, or a parquet path. Ensure column `extraction_prompt` exists.

In [ ]:
import pandas as pd

CATALOG = "shm"  # TODO: update to your catalog
SCHEMA = "default"

# From Unity Catalog table (preferred on Databricks)
les_mis_spark = spark.table(f"{CATALOG}.{SCHEMA}.les_mis_w_prompt")
les_mis = les_mis_spark.toPandas()

# Or from parquet if you have a path:
# les_mis = pd.read_parquet("/path/to/les_mis_w_prompt.parquet")

print(f"Rows: {len(les_mis)}")
print(les_mis.columns.tolist())

## 3. External model configuration

Point to an OpenAI-compatible endpoint (e.g. Azure OpenAI, or a Databricks external-model endpoint). Use env vars or Databricks secrets.

In [ ]:
import os

# Example: Azure OpenAI
# EXTERNAL_MODEL_BASE_URL = "https://your-resource.openai.azure.com/openai/deployments/your-deployment"
# EXTERNAL_MODEL_API_KEY = os.getenv("AZURE_OPENAI_KEY") or dbutils.secrets.get("scope", "key")

# Example: Databricks external model endpoint (get base URL from workspace)
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
workspace_url = w.config.host

EXTERNAL_MODEL_BASE_URL = os.getenv("EXTERNAL_MODEL_BASE_URL", f"{workspace_url}/serving-endpoints/your-external-endpoint/invocations")
EXTERNAL_MODEL_API_KEY = os.getenv("EXTERNAL_MODEL_API_KEY") or (dbutils.secrets.get("scope", "key") if 'dbutils' in dir() else os.getenv("DATABRICKS_TOKEN"))
EXTERNAL_MODEL_NAME = os.getenv("EXTERNAL_MODEL_NAME", "gpt-4o-mini")

print(f"Base URL: {EXTERNAL_MODEL_BASE_URL[:50]}..." if len(EXTERNAL_MODEL_BASE_URL) > 50 else f"Base URL: {EXTERNAL_MODEL_BASE_URL}")

## 4. Ray remote task: call external model for a batch of prompts

Each Ray task receives a list of prompts and returns model responses. Uses the OpenAI client with a custom base URL for compatibility.

In [ ]:
from openai import OpenAI
import ray


@ray.remote
def call_external_model_batch(prompts: list[str], base_url: str, api_key: str, model: str) -> list[dict]:
    """Call OpenAI-compatible endpoint for a batch of prompts. Returns list of {prompt, response, error}."""
    client = OpenAI(base_url=base_url.rstrip("/") + "/", api_key=api_key)
    results = []
    for prompt in prompts:
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt[:8000]}],  # truncate long prompts
                max_tokens=500,
            )
            text = resp.choices[0].message.content if resp.choices else ""
            results.append({"prompt": prompt[:200], "response": text, "error": None})
        except Exception as e:
            results.append({"prompt": prompt[:200], "response": None, "error": str(e)})
    return results

print("Remote task defined.")

## 5. Run batch inference with Ray

Split the Les Mis dataframe into chunks and run in parallel. Tune `chunk_size` and `max_parallel` for your endpoint rate limits.

In [ ]:
import time

prompt_col = "extraction_prompt"  # from Les Mis data prep
if prompt_col not in les_mis.columns:
    prompt_col = les_mis.columns[0]  # fallback

prompts = les_mis[prompt_col].astype(str).tolist()
chunk_size = 10  # prompts per Ray task
chunks = [prompts[i : i + chunk_size] for i in range(0, len(prompts), chunk_size)]

start = time.perf_counter()
futures = [
    call_external_model_batch.remote(
        c, EXTERNAL_MODEL_BASE_URL, EXTERNAL_MODEL_API_KEY, EXTERNAL_MODEL_NAME
    )
    for c in chunks
]
all_results = ray.get(futures)
elapsed = time.perf_counter() - start

# Flatten and attach back to dataframe
flat = [r for batch in all_results for r in batch]
les_mis["external_model_response"] = [r["response"] for r in flat]
les_mis["external_model_error"] = [r["error"] for r in flat]

print(f"Processed {len(les_mis)} rows in {elapsed:.2f}s ({len(les_mis)/elapsed:.1f} rows/s)")
display(les_mis[[prompt_col, "external_model_response", "external_model_error"]].head(10))

## 6. Optional: save results

Persist the results for comparison with AI_QUERY or other ai_functions runs.

In [ ]:
# Example: write back to a table or volume
# les_mis.to_parquet("/Volumes/shm/default/llm_profiling/les_mis_ray_external.parquet")
# spark.createDataFrame(les_mis).write.saveAsTable("shm.default.les_mis_ray_external")
print("Done. Use the cell above to save results.")